# GW170817: sky localization by triangulation

This tutorial reconstructs the basic timing geometry of **GW170817** with LIGO Hanford (H), LIGO Livingston (L), and Virgo (V).

Event time: **2017-08-17 12:41:04.43 UTC**

Source position:

$$
\mathrm{RA}\simeq197.45^\circ,\qquad
\mathrm{Dec}\simeq-23.38^\circ.
$$

We focus on geometry: one baseline gives a ring; two independent baselines give intersections. We do **not** derive timing uncertainty from SNR.

## 1. Time-delay equation

For a plane wave,

$$
\Phi=\omega t-\mathbf{k}\cdot\mathbf r.
$$

At two detectors,

$$
\omega t_1-\mathbf{k}\cdot\mathbf r_1=\Phi_0,
$$

$$
\omega t_2-\mathbf{k}\cdot\mathbf r_2=\Phi_0.
$$

Subtracting,

$$
t_2-t_1=\frac{(\mathbf r_2-\mathbf r_1)\cdot\hat{\mathbf k}}{c}.
$$

RA/Dec gives $\hat{\mathbf n}$ pointing **toward the source**, whereas the GW propagates along $\hat{\mathbf k}=-\hat{\mathbf n}$. Therefore

$$
\Delta t_{21}
=
-\frac{(\mathbf r_2-\mathbf r_1)\cdot\hat{\mathbf n}}{c}.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

c = 299792458.0

ra_true = 197.45
dec_true = -23.38

def source_vector(ra_deg, dec_deg):
    ra, dec = np.deg2rad([ra_deg, dec_deg])
    return np.array([np.cos(dec)*np.cos(ra),
                     np.cos(dec)*np.sin(ra),
                     np.sin(dec)])

n_eci = source_vector(ra_true, dec_true)
print(n_eci)

## 2. Real H, L, V detector sites

We use approximate detector latitudes and longitudes and a spherical Earth. This is sufficient for a clean undergraduate triangulation exercise.

In [ ]:
R = 6371e3

sites = {
    "H": (46.455, -119.408),
    "L": (30.563,  -90.774),
    "V": (43.631,   10.504),
}

def ecef(lat_deg, lon_deg):
    lat, lon = np.deg2rad([lat_deg, lon_deg])
    return R*np.array([np.cos(lat)*np.cos(lon),
                       np.cos(lat)*np.sin(lon),
                       np.sin(lat)])

r = {name: ecef(*coords) for name, coords in sites.items()}

for a,b in [("H","L"),("H","V"),("L","V")]:
    B = np.linalg.norm(r[a]-r[b])
    print(f"{a}-{b}: {B/1e3:.0f} km; B/c = {1e3*B/c:.2f} ms")

## 3. Rotate the source into the Earth-fixed frame

RA/Dec is fixed on the celestial sphere, while H, L, and V rotate with Earth. We therefore rotate the source direction into the Earth-fixed frame at the event time.

The compact GMST calculation below avoids requiring `astropy`.

In [ ]:
event = datetime(2017,8,17,12,41,4,430000,tzinfo=timezone.utc)

def julian_date(dt):
    y, m = dt.year, dt.month
    d = dt.day + (dt.hour + (dt.minute + (dt.second+dt.microsecond/1e6)/60)/60)/24
    if m <= 2:
        y -= 1; m += 12
    A = y//100
    B = 2-A+A//4
    return int(365.25*(y+4716))+int(30.6001*(m+1))+d+B-1524.5

def gmst_deg(dt):
    JD = julian_date(dt)
    T = (JD-2451545.0)/36525
    return (280.46061837 + 360.98564736629*(JD-2451545.0)
            + 0.000387933*T*T - T**3/38710000) % 360

g = np.deg2rad(gmst_deg(event))
Rz = np.array([[ np.cos(g), np.sin(g), 0],
               [-np.sin(g), np.cos(g), 0],
               [0,0,1]])

n = Rz @ n_eci
print(f"GMST = {np.rad2deg(g):.3f} deg")
print("Earth-fixed source vector =", n)

## 4. GW170817 arrival-time differences

Define

$$
\Delta t_{AB}=t_A-t_B.
$$

Then

$$
\Delta t_{AB}
=
-\frac{(\mathbf r_A-\mathbf r_B)\cdot\hat{\mathbf n}}{c}.
$$

GW170817 reached Virgo first, Livingston about $22$ ms later, and Hanford about another $3$ ms later.

In [ ]:
def delay(A,B,nvec=n):
    return -np.dot(r[A]-r[B], nvec)/c

dt_HL = delay("H","L")
dt_LV = delay("L","V")
dt_HV = delay("H","V")

print(f"t_H - t_L = {1e3*dt_HL:+.3f} ms")
print(f"t_L - t_V = {1e3*dt_LV:+.3f} ms")
print(f"t_H - t_V = {1e3*dt_HV:+.3f} ms")
print()
print("Check:", 1e3*(dt_HL+dt_LV), "ms =", 1e3*dt_HV, "ms")

Only two delays are independent:

$$
\Delta t_{HV}=\Delta t_{HL}+\Delta t_{LV}.
$$

Three arrival times minus one arbitrary common arrival time leave two independent timing observables.

## 5. One baseline gives a ring

For Hanford and Livingston,

$$
\Delta t_{HL}
=
-\frac{\mathbf B_{HL}\cdot\hat{\mathbf n}}{c},
\qquad
\mathbf B_{HL}=\mathbf r_H-\mathbf r_L.
$$

If $\theta$ is the angle to the baseline,

$$
|\cos\theta|=\frac{c|\Delta t_{HL}|}{B_{HL}}.
$$

The delay fixes $\theta$, but not the azimuth around the baseline. Therefore one detector pair gives a **ring on the sky**.

In [ ]:
BHL = r["H"]-r["L"]
theta = np.arccos(np.clip(c*abs(dt_HL)/np.linalg.norm(BHL),-1,1))
print(f"|t_H-t_L| = {1e3*abs(dt_HL):.3f} ms")
print(f"theta = {np.rad2deg(theta):.2f} deg")

## 6. Plot the H–L timing ring

For each trial RA/Dec we calculate the predicted H–L delay. The contour at the observed delay is the timing ring.

In [ ]:
ra = np.linspace(0,360,721)
dec = np.linspace(-90,90,361)
RA, DEC = np.meshgrid(ra,dec)

rar = np.deg2rad(RA); decr = np.deg2rad(DEC)
Neci = np.stack([np.cos(decr)*np.cos(rar),
                 np.cos(decr)*np.sin(rar),
                 np.sin(decr)],axis=-1)
Necef = Neci @ Rz.T

def delay_map(A,B):
    return -(Necef @ (r[A]-r[B]))/c

DHL = delay_map("H","L")

plt.figure(figsize=(10,5))
plt.contour(RA,DEC,1e3*DHL,levels=[1e3*dt_HL],linewidths=2)
plt.scatter(ra_true,dec_true,marker="*",s=140,label="GW170817")
plt.xlim(360,0); plt.ylim(-90,90)
plt.xlabel("RA [deg]"); plt.ylabel("Dec [deg]")
plt.grid(alpha=.25); plt.legend()
plt.title("Hanford–Livingston timing ring")
plt.show()

## 7. Add Virgo

The independent Livingston–Virgo constraint is

$$
\Delta t_{LV}
=
-\frac{(\mathbf r_L-\mathbf r_V)\cdot\hat{\mathbf n}}{c}.
$$

The source must lie on **both** rings.

In [ ]:
DLV = delay_map("L","V")

plt.figure(figsize=(10,5))
plt.contour(RA,DEC,1e3*DHL,levels=[1e3*dt_HL],linewidths=2)
plt.contour(RA,DEC,1e3*DLV,levels=[1e3*dt_LV],linewidths=2)
plt.scatter(ra_true,dec_true,marker="*",s=140,label="GW170817")
plt.xlim(360,0); plt.ylim(-90,90)
plt.xlabel("RA [deg]"); plt.ylabel("Dec [deg]")
plt.grid(alpha=.25); plt.legend()
plt.title("Two independent timing rings")
plt.show()

Thus,

$$
\Delta t_{HL}
\rightarrow
\text{one ring},
$$

while

$$
\Delta t_{HL}+\Delta t_{LV}
\rightarrow
\text{ring intersections}.
$$

Timing alone generally leaves a mirror ambiguity associated with reflection through the plane containing the three detector sites. Real localization also uses relative amplitudes, phases, antenna patterns, waveform information, and detector noise.

## 8. Draw all three pairwise rings

The H–V ring is useful visually, but it contains no new independent ideal timing information because

$$
\Delta t_{HV}=\Delta t_{HL}+\Delta t_{LV}.
$$

In [ ]:
DHV = delay_map("H","V")

plt.figure(figsize=(10,5))
plt.contour(RA,DEC,1e3*DHL,levels=[1e3*dt_HL],linewidths=2)
plt.contour(RA,DEC,1e3*DLV,levels=[1e3*dt_LV],linewidths=2)
plt.contour(RA,DEC,1e3*DHV,levels=[1e3*dt_HV],linewidths=2)
plt.scatter(ra_true,dec_true,marker="*",s=140,label="GW170817")
plt.xlim(360,0); plt.ylim(-90,90)
plt.xlabel("RA [deg]"); plt.ylabel("Dec [deg]")
plt.grid(alpha=.25); plt.legend()
plt.title("H–L, L–V, and H–V timing rings")
plt.show()

## 9. From a ring to an annulus

An exact delay gives an infinitely thin ring. A real measurement has finite uncertainty,

$$
\Delta t_{\rm measured}\pm\delta(\Delta t).
$$

From

$$
\Delta t=\frac{B}{c}\cos\theta,
$$

we obtain

$$
\delta\theta
\simeq
\frac{c\,\delta(\Delta t)}
{B|\sin\theta|}.
$$

This calculation itself makes **no assumption about SNR**. The measured timing uncertainty is simply supplied as an input.

SNR affects how accurately the arrival time can be measured, but that is an earlier step which we do not need here.

In [ ]:
# Illustrative width only -- not a fitted GW170817 timing posterior
delta_t = 0.5e-3

mask_HL = np.abs(DHL-dt_HL) <= delta_t
mask_LV = np.abs(DLV-dt_LV) <= delta_t
intersection = mask_HL & mask_LV

plt.figure(figsize=(10,5))
plt.contour(RA,DEC,mask_HL.astype(float),levels=[0.5],linewidths=2)
plt.contour(RA,DEC,mask_LV.astype(float),levels=[0.5],linewidths=2)
plt.contourf(RA,DEC,intersection.astype(float),levels=[0.5,1.5],alpha=.3)
plt.scatter(ra_true,dec_true,marker="*",s=140,label="GW170817")
plt.xlim(360,0); plt.ylim(-90,90)
plt.xlabel("RA [deg]"); plt.ylabel("Dec [deg]")
plt.grid(alpha=.25); plt.legend()
plt.title("Finite timing uncertainty: annuli and their intersection")
plt.show()

The $0.5$ ms width above is **only an illustrative choice** to make the geometry visible.

$$
\text{exact delay}\rightarrow\text{ring},
$$

$$
\text{finite timing uncertainty}\rightarrow\text{annulus},
$$

$$
\text{multiple annuli}\rightarrow\text{finite sky patches}.
$$

## 10. Why Virgo mattered

For GW170817, adding Virgo changed the localization dramatically. The full analysis used more than timing alone.

In particular, Virgo saw no strong signal even though GW170817 was loud in the two LIGO detectors. Because the detector response depends on sky direction and polarization, Virgo's weak response itself supplied strong directional information.

The final HLV localization was about $28~\mathrm{deg}^2$ at 90% credibility.

Our timing-ring exercise is meant to explain the **geometry**, not reproduce the full Bayesian sky map.

## 11. Exercises

**1. Remove Virgo.** Plot only H–L. Why can one delay not determine both RA and Dec?

**2. Move the source.** Change `ra_true` and `dec_true`, recompute the delays, and verify that the rings move through the new source.

**3. Change the timing width.** Try `2e-3`, `1e-3`, `0.5e-3`, and `0.2e-3` s. Explain the result using

$$
\delta\theta\simeq\frac{c\,\delta(\Delta t)}{B|\sin\theta|}.
$$

**4. Check the third delay.** Verify numerically that

$$
\Delta t_{HV}=\Delta t_{HL}+\Delta t_{LV}.
$$

**5. Change the network geometry.** Move Virgo's longitude and see how the angle between baselines changes the ring intersections.

## Take-home message

$$
\boxed{
\text{arrival times}
\rightarrow
\text{time differences}
\rightarrow
\text{timing rings}
\rightarrow
\text{ring intersections}
\rightarrow
\text{sky localization}
}
$$

GW170817 is an especially useful example because it shows directly why a geographically extended three-detector network is much more powerful than two nearly aligned detectors.